In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Instacart Market Basket Analysis

## Feature Engineering

This notebook builds the full feature matrix from the cleaned parquet tables. It is self-contained: it reconstructs the candidate set from prior history, then attaches three feature families, each built strictly from prior orders so the final order is touched only to read the label.

The candidate-set construction below duplicates logic from the EDA notebook. This is a deliberate shortcut for reproducibility, each notebook runs top-to-bottom from raw inputs without depending on the other's in-memory state.

In [2]:
PROCESSED_DIR = Path("../data/processed")

orders = pd.read_parquet(PROCESSED_DIR / "orders.parquet")
order_products_prior = pd.read_parquet(PROCESSED_DIR / "order_products_prior.parquet")
order_products_train = pd.read_parquet(PROCESSED_DIR / "order_products_train.parquet")
products = pd.read_parquet(PROCESSED_DIR / "products.parquet")
aisles = pd.read_parquet(PROCESSED_DIR / "aisles.parquet")
departments = pd.read_parquet(PROCESSED_DIR / "departments.parquet")

for name, df in [("orders", orders), ("order_products_prior", order_products_prior),
                 ("order_products_train", order_products_train)]:
    print(f"{name}: {df.shape}")

orders: (3421083, 7)
order_products_prior: (32434489, 4)
order_products_train: (1384617, 4)


### Reconstruct the Candidate Set and Label

The candidate set is every distinct (user, product) pair from a user's prior history, for the train-cohort users whose final order is revealed. The label $y_{u,p}$ is 1 if that product appears in the user's final order. This logic is duplicated from EDA by design.

In [3]:
train_order_ids = orders.loc[orders["eval_set"] == "train", ["order_id", "user_id"]]
prior_order_ids = orders.loc[orders["eval_set"] == "prior", ["order_id", "user_id"]]

prior_user_products = (
    order_products_prior.merge(prior_order_ids, on="order_id")[["user_id", "product_id"]]
    .drop_duplicates()
)

train_user_products = (
    order_products_train.merge(train_order_ids, on="order_id")[["user_id", "product_id"]]
    .drop_duplicates()
)

candidates = prior_user_products.copy()
candidates["y"] = candidates.merge(
    train_user_products.assign(in_final=1),
    on=["user_id", "product_id"],
    how="left"
)["in_final"].fillna(0).astype("int8").values

print(f"Candidate set size: {len(candidates):,}")
print(f"Positive rate: {candidates['y'].mean():.4f}")

Candidate set size: 13,307,953
Positive rate: 0.0623


### User-Level Features $\phi_u$

Six features forming each user's behavioural fingerprint, independent of product, all computed strictly from prior orders. The interval variance carries the censoring handling motivated in EDA, and the variance-undefined case (users with a single inter-order gap) is handled explicitly rather than left as NaN.

In [4]:
prior_op = order_products_prior.merge(prior_order_ids, on="order_id")
n_orders = prior_order_ids.groupby("user_id")["order_id"].nunique().rename("N_u")
basket_per_order = prior_op.groupby("order_id").size().rename("basket_size").reset_index()
basket_per_order["user_id"] = basket_per_order["order_id"].map(
    prior_order_ids.set_index("order_id")["user_id"]
)
mean_basket = basket_per_order.groupby("user_id")["basket_size"].mean().rename("mean_basket_size_u")

print(n_orders.describe())
print(mean_basket.describe())

count    206209.000000
mean         15.590367
std          16.654774
min           3.000000
25%           5.000000
50%           9.000000
75%          19.000000
max          99.000000
Name: N_u, dtype: float64
count    206209.000000
mean          9.951586
std           5.863570
min           1.000000
25%           5.740741
50%           8.933333
75%          13.000000
max          70.250000
Name: mean_basket_size_u, dtype: float64


In [5]:
dspo = orders.loc[orders["eval_set"] == "prior", ["user_id", "days_since_prior_order"]].dropna()

mean_interval = dspo.groupby("user_id")["days_since_prior_order"].mean().rename("mean_interval_u")
var_interval = dspo.groupby("user_id")["days_since_prior_order"].var().rename("var_interval_u")

print(mean_interval.describe())
print(f"\nUsers with NaN variance (only one prior interval): {var_interval.isna().sum():,}")
print(f"Users at the 30-day censoring boundary (mean == 30): {(mean_interval == 30).sum():,}")

count    206209.000000
mean         15.209435
std           7.105644
min           0.000000
25%           9.416667
50%          14.500000
75%          20.285715
max          30.000000
Name: mean_interval_u, dtype: float64

Users with NaN variance (only one prior interval): 0
Users at the 30-day censoring boundary (mean == 30): 8,797


In [6]:

censored_flag = (mean_interval == 30).astype("int8").rename("interval_censored_u")

print(f"Censored users flagged: {censored_flag.sum():,} ({100*censored_flag.mean():.2f}%)")
print(f"\nVariance of interval for censored users (should be near 0, mechanically):")
print(var_interval[mean_interval == 30].describe())

Censored users flagged: 8,797 (4.27%)

Variance of interval for censored users (should be near 0, mechanically):
count    8797.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: var_interval_u, dtype: float64


The censoring flag is justified by direct evidence, not assertion. All 8,797 flagged users (4.27%) have interval variance of exactly 0.0, every recorded gap is identically 30 because the field caps anything larger. Their apparent perfect regularity is the cap mechanically suppressing variation, not genuine behaviour. Without the flag, `var_interval_u = 0` would signal "perfectly regular shopper" to the model for these users, a fabricated signal. The flag lets the model distinguish genuinely-zero-variance users from censored ones, since the raw variance feature alone cannot tell them apart.

In [8]:
# Reorder ratio: fraction of a user's prior line-items that were reorders.
reorder_ratio = prior_op.groupby("user_id")["reordered"].mean().rename("reorder_ratio_u")

# Unique products: distinct products a user has ever purchased in prior history.
unique_products = prior_op.groupby("user_id")["product_id"].nunique().rename("unique_products_u")

print(reorder_ratio.describe())
print(unique_products.describe())

count    206209.000000
mean          0.432249
std           0.212144
min           0.000000
25%           0.267857
50%           0.428571
75%           0.595745
max           0.989529
Name: reorder_ratio_u, dtype: float64
count    206209.000000
mean         64.536238
std          56.592339
min           1.000000
25%          25.000000
50%          48.000000
75%          86.000000
max         726.000000
Name: unique_products_u, dtype: float64


In [9]:
# Assemble the user-level feature table, one row per user.
user_features = pd.concat(
    [n_orders, mean_basket, mean_interval, var_interval, censored_flag,
     reorder_ratio, unique_products],
    axis=1
)

print(f"User feature table shape: {user_features.shape}")
print(f"\nNull audit:")
print(user_features.isnull().sum())
print(f"\nDtypes:")
print(user_features.dtypes)

User feature table shape: (206209, 7)

Null audit:
N_u                    0
mean_basket_size_u     0
mean_interval_u        0
var_interval_u         0
interval_censored_u    0
reorder_ratio_u        0
unique_products_u      0
dtype: int64

Dtypes:
N_u                      int64
mean_basket_size_u     float64
mean_interval_u        float32
var_interval_u         float32
interval_censored_u       int8
reorder_ratio_u        float64
unique_products_u        int64
dtype: object


In [11]:
# Align dtypes for consistency before checkpointing. Counts fit comfortably in int32,
# rates and means in float32, the flag stays int8.
user_features["N_u"] = user_features["N_u"].astype("int32")
user_features["unique_products_u"] = user_features["unique_products_u"].astype("int32")
user_features["mean_basket_size_u"] = user_features["mean_basket_size_u"].astype("float32")
user_features["reorder_ratio_u"] = user_features["reorder_ratio_u"].astype("float32")

print(user_features.dtypes)

N_u                      int32
mean_basket_size_u     float32
mean_interval_u        float32
var_interval_u         float32
interval_censored_u       int8
reorder_ratio_u        float32
unique_products_u        int32
dtype: object


In [12]:
USER_FEATURES_PATH = PROCESSED_DIR / "user_features.parquet"

user_features_out = user_features.rename_axis("user_id").reset_index()
user_features_out.to_parquet(USER_FEATURES_PATH, index=False)

print(f"Saved user features: {user_features_out.shape} -> {USER_FEATURES_PATH}")
print(f"\nColumns: {list(user_features_out.columns)}")
print(f"user_id present as column: {'user_id' in user_features_out.columns}")

Saved user features: (206209, 8) -> ../data/processed/user_features.parquet

Columns: ['user_id', 'N_u', 'mean_basket_size_u', 'mean_interval_u', 'var_interval_u', 'interval_censored_u', 'reorder_ratio_u', 'unique_products_u']
user_id present as column: True


The final two user features both span wide ranges, confirming they carry real between-user signal. Reorder ratio (median 0.43) varies from users who almost never repeat to users who almost always do, a direct measure of individual repeat propensity. Unique products is heavily right-skewed (median 48, tail to 726), the per-user driver behind the large candidate set found in EDA. Assembled into one row per user, the seven-column table passes a clean null audit, no missing values across any feature, and dtypes are aligned to int32, float32, and int8 for a predictable matrix downstream. The user family is checkpointed to parquet, completing the first of three families and surviving a kernel restart from here.

### Product-Level Features $\phi_p$: Reorder Rate and Why It Needs Correction

The raw product reorder rate $r_p$ cannot be used as a feature directly, because EDA showed it is mechanically biased for thinly-observed products. A product purchased only once can show a reorder rate of exactly 0 and nothing else, since there was no second purchase that could count as a reorder, the zero reflects lack of opportunity, not genuine customer behaviour. 3,570 products sit at this fake zero, and feeding it to the model would teach it that these products are never rebought when the truth is simply that they were barely observed.

The correction is empirical Bayes shrinkage, which replaces each product's raw rate with a blend of its own observed rate and the global mean, weighted by how much evidence the product has:

$$\hat{r}_p = \frac{n_p}{n_p + k}\,\bar{r}_p + \frac{k}{n_p + k}\,\bar{r}_{\text{global}}$$

Products with many purchases keep their own rate; products with few are pulled toward the global mean, where the global prior is a more reliable estimate than their own noisy one. The strength $k$ is estimated from the data via the Beta-Binomial model rather than chosen by hand, so the amount of correction is determined by how much products genuinely differ from each other relative to sampling noise, not by an arbitrary constant.

In [13]:
product_stats = (
    order_products_prior.groupby("product_id")["reordered"]
    .agg(["mean", "count", "sum"])
    .rename(columns={"mean": "r_p_raw", "count": "n_p", "sum": "n_reorders"})
)

m = order_products_prior["reordered"].mean()
print(f"Global reorder rate (weighted): {m:.4f}")

Global reorder rate (weighted): 0.5897


In [14]:
stable = product_stats[product_stats["n_p"] >= 30]
s2 = stable["r_p_raw"].var()

k = m * (1 - m) / s2 - 1
print(f"Variance of reorder rate among well-observed products (n>=30): {s2:.5f}")
print(f"Estimated shrinkage strength k: {k:.2f}")

Variance of reorder rate among well-observed products (n>=30): 0.02970
Estimated shrinkage strength k: 7.15


In [15]:
# Apply shrinkage: pull each product's rate toward the global mean by evidence weight.
product_stats["r_p_shrunk"] = (
    product_stats["n_p"] / (product_stats["n_p"] + k) * product_stats["r_p_raw"]
    + k / (product_stats["n_p"] + k) * m
).astype("float32")

# Verify it does what EDA wanted: fake-zero low-count products pulled up, well-observed untouched.
low_count = product_stats[product_stats["n_p"] < 10]
high_count = product_stats[product_stats["n_p"] >= 100]
print(f"Low-count products (n<10): raw mean {low_count['r_p_raw'].mean():.4f} -> shrunk mean {low_count['r_p_shrunk'].mean():.4f}")
print(f"High-count products (n>=100): raw mean {high_count['r_p_raw'].mean():.4f} -> shrunk mean {high_count['r_p_shrunk'].mean():.4f}")
print(f"Products formerly at exactly 0, now shrunk above 0: {((low_count['r_p_raw']==0) & (low_count['r_p_shrunk']>0)).sum():,}")

Low-count products (n<10): raw mean 0.1362 -> shrunk mean 0.3994
High-count products (n>=100): raw mean 0.4817 -> shrunk mean 0.4851
Products formerly at exactly 0, now shrunk above 0: 3,570


#### Conclusion: Shrinkage Outcome

The data estimates a shrinkage strength of $k = 7.15$, meaning a product needs roughly seven purchases before its own reorder rate is trusted equally against the global prior of 0.59. The correction behaves exactly as intended and asymmetrically, which is the point: thinly-observed products are corrected aggressively while well-observed ones are left almost untouched.

Low-count products ($n < 10$) have their mean rate pulled from 0.136 up toward 0.399, while high-count products ($n \geq 100$) move only from 0.482 to 0.485, a third of a percent. All 3,570 products formerly stuck at a fake zero are now corrected above it. The result is a reorder-rate feature that is honest about uncertainty, strong where the evidence supports it and deferential to the global prior where it does not, replacing a feature that would otherwise have fed thousands of false zeros into every model.